In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(
    "../data/silver/clean_transactions.csv"
)

In [3]:
print(df.columns)

Index(['gitOutlet_ID', 'Year', 'Month', 'Distributor_ID', 'SKU_ID',
       'Volume_Liters', 'Total_Bill_Value'],
      dtype='str')


In [4]:
#Convert to datetime
df["date"] = pd.to_datetime(df[["Year", "Month"]].assign(DAY=1))

In [5]:
df["date"] = pd.to_datetime(df["Year"].astype(str) + "-" + df["Month"].astype(str) + "-01")

In [6]:
#create a month column
df["month"] = df["date"].dt.month

In [7]:
df.head()

,gitOutlet_ID,Year,Month,Distributor_ID,SKU_ID,Volume_Liters,Total_Bill_Value,date,month
0,OUT_19886,2024,12,DIST_S_02,SKU_10,5.897879,2177.632359,2024-12-01,12
1,OUT_00837,2024,2,DIST_W_01,SKU_03,20.697364,7244.084814,2024-02-01,2
2,OUT_15438,2025,12,DIST_NW_01,SKU_02,55.101801,13959.108790,2025-12-01,12
3,OUT_12992,2025,1,DIST_C_01,SKU_07,24.063953,15641.548770,2025-01-01,1
4,OUT_12334,2025,5,DIST_C_02,SKU_04,47.769665,15525.158660,2025-05-01,5


In [8]:
#Create base features
features = df.groupby("gitOutlet_ID").agg(
    volume_mean=("Volume_Liters", "mean"),
    volume_sum=("Volume_Liters", "sum"),
    transactions=("Total_Bill_Value", "count")
).reset_index()

#STEP 1 — Create Potential Target using 95th percentile
potential = (
    df
    .groupby("gitOutlet_ID")["Volume_Liters"]
    .quantile(0.95)
    .reset_index()
)

#STEP 2 — Rename Column
potential.columns = [
    "gitOutlet_ID",
    "Potential_Target"
]

#STEP 3 — Merge into Features
features = features.merge(
    potential,
    on="gitOutlet_ID",
    how="left"
)

#STEP 4 — Set target variable
features["target"] = features["Potential_Target"]

In [9]:
#Verify target distribution
print("Target (95th Percentile) Statistics:")
print(features["Potential_Target"].describe())
print(f"\nFeatures shape: {features.shape}")
print(f"Columns: {features.columns.tolist()}")
print("\nFirst few rows:")
features.head()

Target (95th Percentile) Statistics:
count    20000.000000
mean       130.336423
std        136.702568
min          6.720802
25%         51.306909
50%         70.542118
75%        130.151321
max        868.710859
Name: Potential_Target, dtype: float64

Features shape: (20000, 6)
Columns: ['gitOutlet_ID', 'volume_mean', 'volume_sum', 'transactions', 'Potential_Target', 'target']

First few rows:


,gitOutlet_ID,volume_mean,volume_sum,transactions,Potential_Target,target
0,OUT_00001,64.001551,5440.131847,85,296.788863,296.788863
1,OUT_00002,66.211067,6952.162030,105,278.467975,278.467975
2,OUT_00003,64.657867,6724.418178,104,287.079196,287.079196
3,OUT_00004,69.447794,7430.913975,107,294.073228,294.073228
4,OUT_00005,62.228415,6658.440410,107,313.352170,313.352170


In [10]:
#Save features with 95th percentile target
features.to_csv(
    "../data/gold/features.csv",
    index=False
)
print("Features saved to ../data/gold/features.csv")

Features saved to ../data/gold/features.csv
